In [64]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [65]:
MODEL = "llama3.2"

In [66]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [67]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194 

In [68]:
system_prompt = "You are an expert webpage summarizer assistant. Sometimes you throw in some witty commentary. You summarize the webpage as if you are telling a story to a friend. You are concise and to the point. You will respond in Markdown"

def build_user_prompt(website) -> str:
    return f"Here are the contents of the webpage you are summarizing: {website.text}. Please summarize the webpage.  You will respond in Markdown."

In [ ]:
def build_messages(website) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_user_prompt(website)}
    ]

# Using the OpenAI client to summarize the webpage
def summarize_website(website) -> str:
    if not website.text:
        print("The website has no text to summarize.")
        return ""
    
    print("Generating summary...\n")

    response = ollama.chat(model=MODEL, messages=build_messages(website))

    print("Response Object: ", response)

    if not response:
        print("No response from the model.")
        return ""

    print("\nSummary generated!\n")

    return response['message']['content']

### Example usage

In [81]:
ed = Website("https://edwarddonner.com")
try:
    summary = summarize_website(ed)
    print(summary)
except Exception as e:
    print(f"Error: {e}")

Generating summary...
Response Object:  model='llama3.2' created_at='2026-05-08T02:40:47.0867226Z' done=True done_reason='stop' total_duration=4085991900 load_duration=93415700 prompt_eval_count=519 prompt_eval_duration=17436600 eval_count=243 eval_duration=3839894900 message=Message(role='assistant', content="**The Story Behind Ed's Webpage**\n\nIt all starts with a friendly hello from Ed, the co-founder and CTO of Nebula.io, an AI company on a mission to use AI for good. Ed loves tinkering with Large Language Models (LLMs) and is passionate about amateur electronic music production – we're not sure what that entails, but it sounds cool!\n\nAs a seasoned expert in LLMs, Ed has created top-rated Udemy courses that have become best-sellers, with 500,000 students across 194 countries. He's also shared some valuable resources on his website, covering topics like AI Coder, AI Builder, and more.\n\nEd invites you to explore his webpage, which includes a curriculum for becoming a Proficient 

In [62]:
def display_summary(website):
    display(Markdown(summarize_website(website)))

In [ ]:
display_summary(ed)